# Predicción de enfermedad cardíaca: Random Forest

Segundo modelo de la serie.

**Dataset:** registros de pacientes con 11 variables clínicas y la columna objetivo `HeartDisease`
(1 = tiene enfermedad cardíaca, 0 = no tiene).

| Variable | Descripción |
|---|---|
| Age | Edad |
| Sex | Sexo (M / F) |
| ChestPainType | Tipo de dolor de pecho (ASY, ATA, NAP, TA) |
| RestingBP | Presión arterial sistólica en reposo (mm Hg) |
| Cholesterol | Colesterol (mg/dl). **Un valor 0 indica dato faltante** |
| FastingBS | Glucemia en ayunas > 120 mg/dl (1 / 0) |
| RestingECG | Electrocardiograma en reposo (Normal, ST, LVH) |
| MaxHR | Frecuencia cardíaca máxima |
| ExerciseAngina | Angina inducida por ejercicio (Y / N) |
| Oldpeak | Depresión del segmento ST |
| ST_Slope | Pendiente del segmento ST (Up, Flat, Down) |

La métrica pedida por la consigna es **accuracy**.

## Notas de metodología

- Todo el preprocesamiento (imputación, encoding, escalado) está dentro de un **`Pipeline` de scikit-learn**,
  así que se ajusta solo con los datos de train de cada fold. Ningún dato de test influye en el entrenamiento.
- El umbral de decisión (threshold) se elige con **predicciones de validación cruzada sobre train**,
  no mirando el conjunto de test.
- El conjunto de test se usa una única vez, al final, para la evaluación.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
                                     StratifiedKFold, cross_val_predict)
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, auc)
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Descarga de los datos desde Google Drive (compartidos como "Cualquier persona con el enlace")
import os
import gdown

ARCHIVOS = {
"heart_train.csv": "10zgipXXcGiJn0xqGz90JwtVFOgcdl6UO",
}

for nombre, file_id in ARCHIVOS.items():
    if not os.path.exists(nombre):
        gdown.download(id=file_id, output=nombre, quiet=True)

In [ ]:
df = pd.read_csv("heart_train.csv")

X = df.drop(columns="HeartDisease")
y = df["HeartDisease"]

# stratify mantiene la misma proporción de enfermos y sanos en train y en test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=10, stratify=y
)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")
print(f"Proporción de enfermos: {y.mean():.2%}")

## Preprocesamiento

- Random Forest necesita las variables categóricas codificadas (One Hot Encoding).
- No necesita escalar: los árboles dividen por umbrales y no dependen de la escala.
- El colesterol faltante se imputa con el mismo criterio que en la regresión logística
  (promedio por `RestingECG` y `ChestPainType`, ver ese notebook para el análisis).

In [ ]:
class ImputadorColesterol(BaseEstimator, TransformerMixin):
    """Reemplaza Cholesterol == 0 por el promedio del grupo (RestingECG, ChestPainType).

    Si el grupo tiene menos de `min_grupo` datos, usa el promedio por RestingECG;
    si tampoco existe, el promedio general. Los promedios se calculan solo en fit.
    """

    def __init__(self, min_grupo=10, agregar_indicador=True):
        self.min_grupo = min_grupo
        self.agregar_indicador = agregar_indicador

    def fit(self, X, y=None):
        validos = X[X["Cholesterol"] > 0]
        grupos = validos.groupby(["RestingECG", "ChestPainType"])["Cholesterol"].agg(["mean", "size"])
        self.media_grupo_ = grupos.loc[grupos["size"] >= self.min_grupo, "mean"].to_dict()
        self.media_ecg_ = validos.groupby("RestingECG")["Cholesterol"].mean().to_dict()
        self.media_global_ = validos["Cholesterol"].mean()
        return self

    def _imputar(self, fila):
        clave = (fila["RestingECG"], fila["ChestPainType"])
        if clave in self.media_grupo_:
            return self.media_grupo_[clave]
        return self.media_ecg_.get(fila["RestingECG"], self.media_global_)

    def transform(self, X):
        X = X.copy()
        faltante = X["Cholesterol"] == 0
        if self.agregar_indicador:
            X["IsCholesterolMissing"] = faltante.astype(int)
        X["Cholesterol"] = X["Cholesterol"].astype(float)
        if faltante.any():
            X.loc[faltante, "Cholesterol"] = X.loc[faltante].apply(self._imputar, axis=1)
        return X

In [ ]:
NUMERICAS = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
CATEGORICAS = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
BINARIAS = ["FastingBS", "IsCholesterolMissing"]

In [ ]:
preprocesador = ColumnTransformer([
    ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), CATEGORICAS),
], remainder="passthrough", verbose_feature_names_out=False)

pipeline = Pipeline([
    ("imputador", ImputadorColesterol()),
    ("preprocesador", preprocesador),
    ("modelo", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

## Búsqueda de hiperparámetros

In [ ]:
parametros = {
    "modelo__n_estimators": [100, 200, 300],        # Cantidad de árboles. Más de 300 suele ser más lento sin mejorar
    "modelo__max_depth": [5, 10, 15, 20],           # Profundidad máxima: limitarla ayuda a evitar sobreajuste
    "modelo__min_samples_split": [5, 10, 15],       # Mínimo de muestras para dividir un nodo
    "modelo__min_samples_leaf": [2, 4, 6],          # Mínimo de muestras en una hoja: evita hojas muy chicas
    "modelo__max_features": ["sqrt", "log2"],       # Features evaluadas en cada split
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

busqueda = RandomizedSearchCV(
    pipeline,
    param_distributions=parametros,
    n_iter=50,                 # 50 combinaciones de las 216 posibles
    scoring="accuracy",
    cv=cv,
    random_state=42,
    n_jobs=-1,
)
busqueda.fit(X_train, y_train)

mejor_modelo = busqueda.best_estimator_
print(f"Mejores parámetros: {busqueda.best_params_}")
print(f"Mejor accuracy (CV): {busqueda.best_score_:.4f}")

## Evaluación en test

In [ ]:
def evaluar(modelo, X, y, umbral=0.5, titulo=""):
    """Imprime las métricas en test y grafica la matriz de confusión."""
    probas = modelo.predict_proba(X)[:, 1]
    pred = (probas >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()

    print(f"{titulo} (umbral = {umbral:.2f})")
    print(f"  Accuracy:      {accuracy_score(y, pred):.4f}")
    print(f"  Sensibilidad:  {tp / (tp + fn):.4f}   (enfermos detectados)")
    print(f"  Especificidad: {tn / (tn + fp):.4f}   (sanos bien clasificados)")
    print(f"  ROC-AUC:       {roc_auc_score(y, probas):.4f}")
    return probas


def graficar_matriz_confusion(y, pred, titulo):
    plt.figure(figsize=(6, 5))
    sns.heatmap(confusion_matrix(y, pred), annot=True, fmt="d", cmap="Blues", vmin=0,
                xticklabels=["Sano", "Enfermo"], yticklabels=["Sano", "Enfermo"])
    plt.title(titulo)
    plt.xlabel("Predicción")
    plt.ylabel("Real")
    plt.show()


def graficar_roc(y, probas, titulo):
    fpr, tpr, _ = roc_curve(y, probas)
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"Curva ROC (AUC = {auc(fpr, tpr):.4f})")
    plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Clasificador aleatorio")
    plt.xlabel("Tasa de falsos positivos (FPR)")
    plt.ylabel("Tasa de verdaderos positivos (TPR)")
    plt.title(titulo)
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
print(f"Accuracy en train: {accuracy_score(y_train, mejor_modelo.predict(X_train)):.4f}\n")
probas_test = evaluar(mejor_modelo, X_test, y_test, titulo="Test")
graficar_matriz_confusion(y_test, mejor_modelo.predict(X_test), "Matriz de confusión: Random Forest")

In [ ]:
graficar_roc(y_test, probas_test, "Curva ROC: Random Forest")

## Importancia de las variables

In [ ]:
nombres = mejor_modelo.named_steps["preprocesador"].get_feature_names_out()
importancias = pd.Series(mejor_modelo.named_steps["modelo"].feature_importances_, index=nombres).sort_values()

plt.figure(figsize=(9, 7))
importancias.plot(kind="barh")
plt.title("Importancia de las variables: Random Forest")
plt.xlabel("Importancia (reducción media de impureza)")
plt.tight_layout()
plt.show()

## ¿El umbral de decisión impacta en el accuracy?

Busco el umbral que maximiza el accuracy usando **probabilidades de validación cruzada sobre train**.
Elegirlo mirando el test haría que el resultado en test sea optimista.

In [ ]:
probas_cv = cross_val_predict(mejor_modelo, X_train, y_train, cv=cv, method="predict_proba")[:, 1]

umbrales = np.arange(0.05, 0.96, 0.01)
accuracies = [accuracy_score(y_train, (probas_cv >= u).astype(int)) for u in umbrales]
umbral_optimo = umbrales[int(np.argmax(accuracies))]

plt.figure(figsize=(8, 5))
plt.plot(umbrales, accuracies)
plt.axvline(0.5, color="gray", linestyle="--", label="Umbral por defecto (0,5)")
plt.axvline(umbral_optimo, color="darkorange", linestyle="--", label=f"Umbral óptimo ({umbral_optimo:.2f})")
plt.xlabel("Umbral")
plt.ylabel("Accuracy (validación cruzada)")
plt.title("Accuracy según el umbral de decisión")
plt.legend()
plt.show()

print(f"Accuracy CV con umbral 0.50: {accuracy_score(y_train, (probas_cv >= 0.5).astype(int)):.4f}")
print(f"Accuracy CV con umbral {umbral_optimo:.2f}: {max(accuracies):.4f}")

In [ ]:
_ = evaluar(mejor_modelo, X_test, y_test, umbral=0.5, titulo="Test")
print()
_ = evaluar(mejor_modelo, X_test, y_test, umbral=umbral_optimo, titulo="Test")

Si la mejora con el umbral óptimo es chica, conviene quedarse con 0,5: un umbral muy ajustado
a un dataset de este tamaño puede no generalizar.